# 1) CHARGEMENT DES PACKAGES NÉCESSAIRES

In [12]:
# Standard library
import io
import re
import math
# Scientific stack
import numpy as np
import pandas as pd
# Cloud
import boto3
# Typing
from typing import Optional, Dict, List, Iterable
from typing import Dict, List, Iterable, Optional

import io, time
import pandas as pd
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

import io
import time
import pandas as pd
import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

# 2) DÉFINITION DES FONCTIONS

## 2.1 Lecture S3 : parquet -> DataFrame

In [13]:

def read_parquet_from_s3(
    bucket: str,
    key: str,
    *,
    endpoint_url: str,
    aws_access_key_id: str,
    aws_secret_access_key: str,
    verify: bool = False,
    columns: list[str] | None = None,   # <- projection colonnes (pyarrow recommandé)
    engine: str | None = "pyarrow",     # <- pour activer la projection
    max_attempts: int = 6,
    base_sleep: float = 1.0,
    fallback_download: bool = True      # <- bascule sur “download then read” si throttled
) -> pd.DataFrame:
    """
    Lit un Parquet depuis S3/MinIO en gérant les ralentissements (SlowDownRead).
    - columns: liste de colonnes à rapatrier (si None -> tout)
    - engine: 'pyarrow' recommandé pour la projection de colonnes
    - fallback_download: si True, tente un téléchargement local si la lecture stream échoue
    """
    s3_client = boto3.client(
        "s3",
        endpoint_url=endpoint_url,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        verify=verify,
        config=Config(
            retries={"max_attempts": max_attempts, "mode": "adaptive"},
            max_pool_connections=10,
            read_timeout=120,
            connect_timeout=30,
        ),
    )

    attempt = 0
    while True:
        try:
            obj = s3_client.get_object(Bucket=bucket, Key=key)
            return pd.read_parquet(io.BytesIO(obj["Body"].read()),
                                   columns=columns, engine=engine)
        except ClientError as e:
            code = e.response.get("Error", {}).get("Code", "")
            retryable = code in {"SlowDown", "SlowDownRead", "RequestTimeout", "503", "503SlowDown"}
            attempt += 1
            if retryable and attempt < max_attempts:
                sleep_s = min(base_sleep * (2 ** (attempt-1)), 30)
                print(f"⚠️ {code}: tentative {attempt}/{max_attempts} — nouvelle tentative dans ~{sleep_s:.1f}s…")
                time.sleep(sleep_s)
                continue
            # Dernier recours: on bascule sur un téléchargement local
            if fallback_download:
                print(f"ℹ️ Passage en mode 'download then read' (raison: {code or 'échec lecture stream'})")
                return download_then_read_parquet(
                    bucket, key,
                    endpoint_url=endpoint_url,
                    aws_access_key_id=aws_access_key_id,
                    aws_secret_access_key=aws_secret_access_key,
                    verify=verify,
                    columns=columns,
                    engine=engine,
                )
            raise


## 2.2 Détection de bases "répétées" (colonnes doublons logiques)

In [14]:

def _base_name(col: str) -> str:
    """
    Normalise un nom de colonne pour détecter les répétitions logiques.
    - retire '(...)' final (ex: '(code 263)'),
    - uniformise séparateurs, compacte espaces, met en minuscules.
    """
    s = str(col).strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)            # retire '(...)' en fin
    s = re.sub(r"[_\-|./]+", " ", s)                  # uniformise séparateurs
    s = re.sub(r"\s+", " ", s).strip()                # compacte espaces
    return s.lower()

def find_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> Dict[str, List[str]]:
    """Regroupe les colonnes par 'base' et retourne les bases ayant > 1 colonne."""
    groups: Dict[str, List[str]] = {}
    for c in df.columns:
        if c in exclude:
            continue
        base = _base_name(c)
        groups.setdefault(base, []).append(c)
    return {base: cols for base, cols in groups.items() if len(cols) > 1}

def report_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> pd.DataFrame:
    """Affiche un résumé et renvoie un tableau des bases répétées."""
    rep = find_repeated_columns(df, exclude=exclude)
    if not rep:
        print("✅ Aucune colonne répétée détectée.")
        return pd.DataFrame(columns=["base", "nb_colonnes", "colonnes"])
    print(f"🔎 Bases répétées détectées: {len(rep)}")
    rows = []
    for base, cols in sorted(rep.items()):
        print(f"  • {base} -> {cols}")
        rows.append({"base": base, "nb_colonnes": len(cols), "colonnes": cols})
    return (pd.DataFrame(rows)
              .sort_values(["nb_colonnes", "base"], ascending=[False, True])
              .reset_index(drop=True))



## 2.3 Extraction YEAR/MONTH depuis les colonnes date

In [15]:

def year_from_source_file(text: str) -> Optional[int]:
    """
    Extrait l'année depuis 'MMYYYY' (ex: '012015.xlsx') ou '...YYYY...'.
    Retourne l'année (int) ou None.
    """
    s = str(text)
    m = re.search(r'(?<!\d)(0[1-9]|1[0-2])((?:19|20)\d{2})(?!\d)', s)
    if m:
        return int(m.group(2))
    m = re.search(r'(?:19|20)\d{2}', s)
    return int(m.group(0)) if m else None

def extract_years_series(src: pd.Series) -> pd.Series:
    """Année par ligne à partir de SOURCE_FICHIER."""
    return src.map(year_from_source_file)

def extract_years_unique(src: pd.Series) -> list[int]:
    """Liste triée des années uniques détectées dans SOURCE_FICHIER."""
    years = extract_years_series(src).dropna().astype(int).unique().tolist()
    years.sort()
    return years

def year_month_from_source_file(s: pd.Series) -> pd.DataFrame:
    """SOURCE_FICHIER 'MMYYYY.xlsx' -> YEAR, MONTH (Int64)."""
    ss = s.astype("string").str.strip()
    m = ss.str.extract(r'^\s*(0[1-9]|1[0-2])\s*((?:19|20)\d{2})', expand=True)
    month = pd.to_numeric(m[0], errors="coerce")
    year  = pd.to_numeric(m[1], errors="coerce")
    return pd.DataFrame({"YEAR": year.astype("Int64"), "MONTH": month.astype("Int64")})

def year_month_from_date_collecte(s: pd.Series) -> pd.DataFrame:
    """DATE_COLLECTE 'YYYY-MM-DD' -> YEAR, MONTH (Int64)."""
    dt = pd.to_datetime(s, errors="coerce", utc=False)
    return pd.DataFrame({"YEAR": dt.dt.year.astype("Int64"), "MONTH": dt.dt.month.astype("Int64")})



## 2.4 Consolidation haute performance des colonnes répétées

In [16]:

def _filled_mask(block: pd.DataFrame, treat_zero_as_filled: bool) -> np.ndarray:
    """
    Matrice bool (n_rows, n_cols): True si cellule 'renseignée'.
    - numériques: non-NaN (et != 0 si treat_zero_as_filled=False)
    - non-numériques: non-NaN et non vide après strip/lower, pas dans {'na','nan','none'}
    """
    n, m = block.shape
    filled = np.zeros((n, m), dtype=bool)
    for j, col in enumerate(block.columns):
        s = block[col]
        if pd.api.types.is_numeric_dtype(s):
            ok = s.notna().to_numpy()
            if not treat_zero_as_filled:
                ok &= (s.to_numpy() != 0)
        else:
            v = s.astype("object").to_numpy()
            ok = pd.notna(v)
            tmp = np.empty_like(v, dtype=object)
            if ok.any():
                tmp[ok] = np.char.lower(np.char.strip(v[ok].astype(str)))
                bad_ok = (tmp[ok] == "") | (tmp[ok] == "na") | (tmp[ok] == "nan") | (tmp[ok] == "none")
                bad = np.zeros(ok.shape, dtype=bool); bad[ok] = bad_ok
                ok &= ~bad
        filled[:, j] = ok
    return filled

def _coalesce_first_non_null(block: pd.DataFrame, treat_zero_as_filled: bool) -> pd.Series:
    """Retourne, par ligne, la 1ère valeur renseignée (vectorisé via argmax)."""
    if block.shape[1] == 1:
        s = block.iloc[:, 0]
        if pd.api.types.is_numeric_dtype(s) and not treat_zero_as_filled:
            return s.where(s.notna() & (s != 0))
        return s
    filled = _filled_mask(block, treat_zero_as_filled)
    any_filled = filled.any(axis=1)
    first_idx = filled.argmax(axis=1)
    arr = block.to_numpy(copy=False)
    rows = np.arange(arr.shape[0])
    out_vals = np.empty(arr.shape[0], dtype=object)
    out_vals[:] = pd.NA
    out_vals[any_filled] = arr[rows[any_filled], first_idx[any_filled]]
    return pd.Series(out_vals, index=block.index, dtype="object")


def _sum_numeric_block(block: pd.DataFrame) -> pd.Series:
    """
    Somme ligne par ligne (axis=1) après conversion numérique.
    - Non numériques -> NaN (ignorées dans la somme)
    - min_count=1 : si tout est NaN sur la ligne => résultat NaN (pas 0)
    """
    # Conversion colonne par colonne pour éviter une coercition coûteuse sur tout le DataFrame
    to_sum = {}
    for c in block.columns:
        to_sum[c] = pd.to_numeric(block[c], errors="coerce")
    num_block = pd.DataFrame(to_sum, index=block.index)
    return num_block.sum(axis=1, min_count=1)


def consolidate_all_years_and_drop_sources(
    df: pd.DataFrame,
    source_col: str = "SOURCE_FICHIER",
    *,
    mode: str = "sum",            # "sum" | "first_non_null" | "concat"
    sep: str = " | ",
    treat_zero_as_filled: bool = True,
    new_name_fmt: str = "{base}"
) -> pd.DataFrame:
    """
    Consolide les colonnes répétées par 'base' en UNE colonne finale par base,
    en travaillant **année par année** détectée dans `source_col`.
      - mode="sum" : somme numérique ligne par ligne (recommandé si exclusivité violée)
      - mode="first_non_null" : première valeur renseignée (coalesce)
      - mode="concat" : concaténation texte des valeurs remplies
    Puis SUPPRIME systématiquement les colonnes sources dupliquées.
    """
    if source_col not in df.columns:
        raise ValueError(f"Colonne {source_col!r} absente.")
    out = df.copy()

    repeated = find_repeated_columns(out, exclude=(source_col,))
    if not repeated:
        print("ℹ️ Aucune base répétée. Rien à consolider.")
        return out

    years = extract_years_unique(out[source_col])

    def _apply_mode(block: pd.DataFrame) -> pd.Series:
        if mode == "sum":
            return _sum_numeric_block(block)
        elif mode == "first_non_null":
            return _coalesce_first_non_null(block, treat_zero_as_filled)
        elif mode == "concat":
            return _concat_block(block, sep=sep)
        else:
            raise ValueError(f"mode inconnu: {mode!r}")

    if years:
        out["_YEAR_"] = extract_years_series(out[source_col])
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            if new_col not in out.columns:
                out[new_col] = pd.NA
            for y in years:
                mask = out["_YEAR_"].eq(y)
                if not mask.any(): 
                    continue
                block = out.loc[mask, cols]
                out.loc[mask, new_col] = _apply_mode(block)
        out.drop(columns=["_YEAR_"], inplace=True, errors="ignore")
    else:
        # pas d'année détectée -> consolidation globale
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            out[new_col] = _apply_mode(out[cols])

    # supprimer les colonnes sources (codes)
    cols_to_drop = sorted({c for cols in repeated.values() for c in cols})
    out.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    return out








def consolidate_all_years_first_non_null_and_drop_sources(
    df: pd.DataFrame,
    source_col: str = "SOURCE_FICHIER",
    *,
    treat_zero_as_filled: bool = True,
    new_name_fmt: str = "{base}"
) -> pd.DataFrame:
    """
    Consolide les colonnes répétées par 'base' (ex: 'indemnite de logement (code XXX)')
    en 1 seule colonne par base (1ère non nulle), année par année si possible,
    puis SUPPRIME les colonnes sources.
    """
    if source_col not in df.columns:
        raise ValueError(f"Colonne {source_col!r} absente.")
    out = df.copy()
    repeated = find_repeated_columns(out, exclude=(source_col,))
    if not repeated:
        print("ℹ️ Aucune base répétée. Rien à faire.")
        return out

    years = extract_years_unique(out[source_col])
    if years:
        out["_YEAR_"] = extract_years_series(out[source_col])
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            if new_col not in out.columns:
                out[new_col] = pd.NA
            for y in years:
                mask = out["_YEAR_"].eq(y)
                if not mask.any(): 
                    continue
                block = out.loc[mask, cols]
                out.loc[mask, new_col] = _coalesce_first_non_null(block, treat_zero_as_filled)
        out.drop(columns=["_YEAR_"], inplace=True, errors="ignore")
    else:
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            out[new_col] = _coalesce_first_non_null(out[cols], treat_zero_as_filled)

    cols_to_drop = sorted({c for cols in repeated.values() for c in cols})
    out.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    return out


## 2.5 Vérification d’exclusivité

In [17]:
def check_repeated_columns_exclusivity_by_year(
    df: pd.DataFrame,
    *,
    source_col: str = "SOURCE_FICHIER",
    treat_zero_as_filled: bool = True,
    show_examples: int = 10
) -> pd.DataFrame:
    """
    Vérifie que, pour chaque groupe de colonnes répétées (même base),
    et pour chaque ANNÉE extraite de SOURCE_FICHIER, AU PLUS UNE colonne du groupe est renseignée par ligne.
    
    - Affiche un message d'alerte si des violations existent (>=2 colonnes remplies).
    - Retourne un DataFrame récapitulatif avec (base, year, violations, rows_in_year, columns).
    - Montre un petit échantillon des lignes violant la règle.
    """
    if source_col not in df.columns:
        raise KeyError(f"Colonne {source_col!r} absente.")

    # 1) Détecter les groupes "répétés"
    repeated = find_repeated_columns(df, exclude=(source_col,))
    if not repeated:
        print("✅ Aucune base répétée détectée — rien à vérifier.")
        return pd.DataFrame(columns=["base", "year", "violations", "rows_in_year", "columns"])

    # 2) Extraire l'année par ligne
    years_series = extract_years_series(df[source_col])
    if years_series.isna().all():
        print("⚠️ Aucune année détectée dans SOURCE_FICHIER — contrôle effectué SANS partition par année (global).")
        years_series = pd.Series(["ALL"] * len(df), index=df.index)

    # 3) Préparer (facultatif) l'identifiant pour exemples
    id_cols = [c for c in ["MATRICULE", "MATRICULE||CODE_ORGANISME", source_col] if c in df.columns]
    if not id_cols:
        id_cols = [source_col]

    # 4) Boucle par base + contrôle vectorisé
    rows_summary = []
    any_violation = False

    for base, cols in repeated.items():
        # masque "filled" pour le bloc
        block = df[cols]
        filled = _filled_mask(block, treat_zero_as_filled)   # shape: (n_rows, n_cols)
        nb_filled = filled.sum(axis=1)                       # nb de colonnes remplies par ligne

        # contrôle par année
        for year_val, mask_year in years_series.groupby(years_series).groups.items():
            mask_year = df.index.isin(mask_year)
            n_in_year = int(mask_year.sum())
            if n_in_year == 0:
                continue

            viol_mask = (nb_filled > 1) & mask_year
            n_viol = int(viol_mask.sum())

            rows_summary.append({
                "base": base,
                "year": year_val,
                "violations": n_viol,
                "rows_in_year": n_in_year,
                "columns": cols
            })

            if n_viol > 0:
                any_violation = True
                print(f"\n🚨 Exclusivité violée — base='{base}', année={year_val} : {n_viol} lignes en anomalie (sur {n_in_year}).")
                # exemple de lignes fautives
                ex_cols = list(dict.fromkeys(id_cols + cols))
                print(df.loc[viol_mask, ex_cols].head(show_examples).to_string(index=False))

    # 5) Résumé global
    summary = (pd.DataFrame(rows_summary)
                 .sort_values(["violations", "base", "year"], ascending=[False, True, True])
                 .reset_index(drop=True))

    if not any_violation:
        print("✅ Exclusivité respectée : pour chaque base et chaque année, au plus une colonne est renseignée par ligne.")
    else:
        total_viol = int(summary["violations"].sum()) if not summary.empty else 0
        print(f"\nBilan: {total_viol} violations au total sur l'ensemble des bases/années.")

    return summary


## 2.6 Jointure sur une année à gauche  : (YEAR, MONTH, MATRICULE)

In [18]:


def write_df_to_xlsx_chunked(
    df: pd.DataFrame,
    path: str,
    *,
    sheet_base: str = "data",
    chunk_rows: int = 1_000_000,
    engine: str = "xlsxwriter"
) -> None:
    """
    Écrit un DataFrame volumineux en XLSX, en le découpant en plusieurs onglets
    si nécessaire (limite Excel ≈ 1 048 576 lignes).
    - sheet_base: préfixe des noms d'onglet (data, data_part2, …)
    - chunk_rows: taille de découpe (par défaut ~1 million)
    """
    n = len(df)
    if n == 0:
        # crée un fichier avec un onglet vide (structure)
        with pd.ExcelWriter(path, engine=engine) as xw:
            df.to_excel(xw, sheet_name=sheet_base, index=False)
        return

    parts = math.ceil(n / chunk_rows)
    with pd.ExcelWriter(path, engine=engine) as xw:
        for i in range(parts):
            start = i * chunk_rows
            stop  = min((i + 1) * chunk_rows, n)
            sheet_name = sheet_base if i == 0 else f"{sheet_base}_part{i+1}"
            df.iloc[start:stop].to_excel(xw, sheet_name=sheet_name, index=False)


# -------- Helpers YEAR/MONTH & normalisation matricule --------
def _year_month_from_source_file(s: pd.Series) -> pd.DataFrame:
    ss = s.astype("string").str.strip()
    m = ss.str.extract(r'^\s*(0[1-9]|1[0-2])\s*((?:19|20)\d{2})', expand=True)
    return pd.DataFrame({
        "YEAR":  pd.to_numeric(m[1], errors="coerce").astype("Int64"),
        "MONTH": pd.to_numeric(m[0], errors="coerce").astype("Int64"),
    })

def _year_month_from_date_collecte(s: pd.Series) -> pd.DataFrame:
    dt = pd.to_datetime(s, errors="coerce", utc=False)
    return pd.DataFrame({
        "YEAR":  dt.dt.year.astype("Int64"),
        "MONTH": dt.dt.month.astype("Int64"),
    })

def _normalize_matricule_min(s: pd.Series) -> pd.Series:
    return (s.astype("string")
             .str.strip()
             .str.replace("\u00A0", "", regex=False))

# -------- Fonction principale --------
def merge_union_for_year_with_suffix(
    df_left: pd.DataFrame,      # base 1 (gauche)  : panel_solde_df
    df_right: pd.DataFrame,     # base 2 (droite)  : panel_solde_df_1
    year: int,
    *,
    join_on_month: bool = True,         # True: clé = (YEAR, MONTH, MATRICULE) ; False: (YEAR, MATRICULE)
    suffix_left: str = "_b0",           # suffix uniquement pour les colonnes communes côté gauche
    suffix_right: str = "_b1",          # suffix uniquement pour les colonnes communes côté droite
    dedup_left: bool = False,           # déduplique côté gauche sur la clé (rarement nécessaire)
    dedup_right: bool = True,           # déduplique côté droit sur la clé (souvent utile)
    export_unmatched_prefix: str | None = None,  # ex: "bilan_merge" -> écrit XLSX des non-matchés
    xlsx_chunk_rows: int = 1_000_000,           # découpe par onglet pour gros volumes
    xlsx_engine: str = "xlsxwriter"
) -> pd.DataFrame:
    """
    Fusion 'union' (OUTER JOIN) sur une année :
      - conserve toutes les lignes (gauche + droite),
      - suffixe SEULEMENT les colonnes de noms identiques (_b0 pour base 1, _b1 pour base 2),
      - ajoute un indicateur d’origine: IN_SRC ∈ {'base1','base2','both'} + IN_BASE1/IN_BASE2 (bool),
      - garde toutes les colonnes des deux bases,
      - exporte les non-appariés en XLSX (avec découpe par onglets si volumineux).
    """

    # 1) Copies
    L = df_left.copy()
    R = df_right.copy()

    # 2) Normaliser MATRICULE
    for d in (L, R):
        if "MATRICULE||CODE_ORGANISME" not in d.columns:
            raise KeyError("Il manque 'MATRICULE||CODE_ORGANISME' dans une des bases.")
        d["MATRICULE"] = _normalize_matricule_min(d["MATRICULE||CODE_ORGANISME"])

    # 3) YEAR/MONTH
    if "SOURCE_FICHIER" not in L.columns:
        raise KeyError("La base gauche doit contenir 'SOURCE_FICHIER'.")
    ymL = _year_month_from_source_file(L["SOURCE_FICHIER"])
    L["YEAR"], L["MONTH"] = ymL["YEAR"], ymL["MONTH"]

    if "DATE_COLLECTE" not in R.columns:
        raise KeyError("La base droite doit contenir 'DATE_COLLECTE'.")
    ymR = _year_month_from_date_collecte(R["DATE_COLLECTE"])
    R["YEAR"], R["MONTH"] = ymR["YEAR"], ymR["MONTH"]

    # 4) Filtre année
    L = L.loc[L["YEAR"].eq(year)].copy()
    R = R.loc[R["YEAR"].eq(year)].copy()

    # 5) Clés
    keys = (["YEAR", "MONTH", "MATRICULE"] if join_on_month else ["YEAR", "MATRICULE"])

    # 6) Colonnes communes (hors clés) -> suffixes sélectifs
    overlap = sorted((set(L.columns) & set(R.columns)) - set(keys))
    L.rename(columns={c: f"{c}{suffix_left}"  for c in overlap}, inplace=True)
    R.rename(columns={c: f"{c}{suffix_right}" for c in overlap}, inplace=True)

    # 7) Dédup éventuel
    if dedup_left:
        L = L.sort_values(keys).drop_duplicates(keys, keep="first")
    if dedup_right:
        R = R.sort_values(keys).drop_duplicates(keys, keep="first")

    # 8) OUTER JOIN
    merged = L.merge(
        R,
        on=keys,
        how="outer",
        suffixes=("", ""),   # déjà pré-suffixé
        indicator=True
    )

    # 9) Indicateurs d’origine
    src_map = {"left_only": "base1", "right_only": "base2", "both": "both"}
    merged["IN_SRC"] = merged["_merge"].map(src_map)
    merged["IN_BASE1"] = merged["_merge"].isin(["left_only", "both"])
    merged["IN_BASE2"] = merged["_merge"].isin(["right_only", "both"])

    # 10) Bilan
    print(f"Fusion OUTER {year} (join_on_month={join_on_month}) → {len(merged):,} lignes, {merged.shape[1]} colonnes")
    print("\nBilan _merge :")
    print(merged["_merge"].value_counts(dropna=False))

    # 11) Exports non appariés en XLSX (avec découpe)
    if export_unmatched_prefix:
        left_cols  = [c for c in ["YEAR","MONTH","MATRICULE","SOURCE_FICHIER"] if c in merged.columns]
        right_cols = [c for c in ["YEAR","MONTH","MATRICULE","DATE_COLLECTE"] if c in merged.columns]

        left_only_df  = merged.loc[merged["_merge"]=="left_only",  left_cols]
        right_only_df = merged.loc[merged["_merge"]=="right_only", right_cols]

        # fichiers : <prefix>_<year>_left_only.xlsx / right_only.xlsx
        left_path  = f"{export_unmatched_prefix}_{year}_left_only.xlsx"
        right_path = f"{export_unmatched_prefix}_{year}_right_only.xlsx"

        write_df_to_xlsx_chunked(left_only_df,  left_path,  sheet_base="left_only",
                                 chunk_rows=xlsx_chunk_rows, engine=xlsx_engine)
        write_df_to_xlsx_chunked(right_only_df, right_path, sheet_base="right_only",
                                 chunk_rows=xlsx_chunk_rows, engine=xlsx_engine)

        print("XLSX écrits :")
        print("  ", left_path)
        print("  ", right_path)

    return merged


## 2.7 Faire une vérification pour voir si la sommme pour chaque ligne à problème donne le montant brute pour les colonnes répétés


In [19]:
def get_multi_filled_row_indices(
    df: pd.DataFrame,
    *,
    source_col: str = "SOURCE_FICHIER",
    treat_zero_as_filled: bool = True,
) -> pd.Index:
    """
    Retourne l'ensemble des index de lignes qui violent l'exclusivité
    (>= 2 colonnes remplies) pour AU MOINS une base répétée,
    en respectant le partitionnement par année (extraction via source_col).
    """
    if source_col not in df.columns:
        raise KeyError(f"Colonne {source_col!r} absente.")

    repeated = find_repeated_columns(df, exclude=(source_col,))
    if not repeated:
        return pd.Index([])

    years_series = extract_years_series(df[source_col])
    if years_series.isna().all():
        years_series = pd.Series(["ALL"] * len(df), index=df.index)

    bad_idx_sets: list[pd.Index] = []

    for base, cols in repeated.items():
        block = df[cols]
        filled = _filled_mask(block, treat_zero_as_filled)  # (n_rows, n_cols)
        nb_filled = filled.sum(axis=1)

        # par année
        for _, inds in years_series.groupby(years_series).groups.items():
            mask_year = df.index.isin(inds)
            viol_mask = (nb_filled > 1) & mask_year
            if viol_mask.any():
                bad_idx_sets.append(df.index[viol_mask])

    if not bad_idx_sets:
        return pd.Index([])

    # Union robuste sans union_many (compat toutes versions Pandas)
    bad_union = bad_idx_sets[0]
    for idx in bad_idx_sets[1:]:
        bad_union = bad_union.union(idx)
    return bad_union


In [20]:
def check_sum_all_base_vs_reference_on_index(
    df: pd.DataFrame,
    row_index_filter: pd.Index,
    *,
    reference_col: str = "MONTANT BRUT",
    abs_tol: float = 1e-6,
    rel_tol: float = 0.0,
    show_examples: int = 10,
) -> pd.DataFrame:
    """
    Pour les lignes dont l'index est dans `row_index_filter` :
    - calcule la somme de TOUTES les colonnes numériques (df.select_dtypes number),
    - compare à `reference_col`,
    - renvoie un DataFrame des anomalies (|sum - ref| > tol).
    """
    if reference_col not in df.columns:
        raise KeyError(f"Colonne de référence {reference_col!r} absente.")

    if len(row_index_filter) == 0:
        print("✅ Aucune ligne à tester (aucune exclusivité violée).")
        return pd.DataFrame(columns=["sum_all", "reference", "abs_diff", "rel_diff"])

    sub = df.loc[row_index_filter]

    num_cols = sub.select_dtypes(include=["number"]).columns.tolist()
    if reference_col not in num_cols:
        # on convertit au besoin
        ref_numeric = pd.to_numeric(sub[reference_col], errors="coerce")
    else:
        ref_numeric = sub[reference_col]

    sum_all = sub[num_cols].sum(axis=1, min_count=1)

    abs_diff = (sum_all - ref_numeric).abs()
    # rel_diff prudente : éviter division par 0 -> si ref=0, on passe rel=inf quand abs_diff>0
    rel_diff = abs_diff / ref_numeric.replace(0, pd.NA)

    # tolérance : (abs_diff > abs_tol) ET (rel_diff > rel_tol OU ref=0 & abs_diff>abs_tol)
    bad_mask = (abs_diff > abs_tol) & (rel_diff.fillna(np.inf) > rel_tol)

    out = sub.copy()
    out["sum_all"] = sum_all
    out["reference"] = ref_numeric
    out["abs_diff"] = abs_diff
    out["rel_diff"] = rel_diff

    anomalies = out.loc[bad_mask].sort_values("abs_diff", ascending=False)

    print(f"🔎 Lignes testées: {len(sub):,} | Anomalies (somme ≠ {reference_col}): {len(anomalies):,}")
    if not anomalies.empty:
        print("\n📌 Exemples d'anomalies:")
        print(anomalies.head(show_examples).to_string(index=True))

    return anomalies


# 3) DÉFINITION DES PARAMÈTRES

In [21]:
ENDPOINT_URL = "http://minio.mon-namespace.svc.cluster.local:80"
AWS_KEY      = "wer1Or8j7hXa3QGk2M3M"
AWS_SECRET   = "YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt"
VERIFY_SSL   = False

BUCKET = "admindataanstat"
KEY_PANEL_2 = "Solde/panel_solde_df_2.parquet"          # panel (montants, primes…)
KEY_PANEL_1 = "Solde/panel_solde_df_1_code.parquet"     # infos admin / codes


# 4) APPEL DES FONCTIONS / PIPELINE

## 4.1 Charger panel_solde_df (panel 2)

In [22]:
panel_solde_df = read_parquet_from_s3(
    BUCKET, KEY_PANEL_2,
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    verify=VERIFY_SSL
)
print(f"panel_solde_df: {len(panel_solde_df):,} lignes, {panel_solde_df.shape[1]} colonnes")


panel_solde_df: 15,081,954 lignes, 190 colonnes


## 4.2 Afficher les bases répétées détectées

In [14]:
_ = report_repeated_columns(panel_solde_df)

🔎 Bases répétées détectées: 4
  • differentiel familial -> ['differentiel familial (code 221)', 'differentiel familial (code 422)']
  • indemnite de logement -> ['indemnite de logement (code 263)', 'indemnite de logement (code 271)', 'indemnite de logement (code 265)']
  • participation judicature -> ['participation judicature (code 440)', 'participation judicature (code 441)', 'participation judicature (code 442)', 'participation judicature (code 443)']
  • responsabilite tresor -> ['responsabilite tresor (code 487)', 'responsabilite tresor (code 493)']


## 4.3 Vérifier l'exclusivité (au plus une colonne renseignée par base & par année)

In [15]:
excl_summary = check_repeated_columns_exclusivity_by_year(
        panel_solde_df,
        source_col="SOURCE_FICHIER",
        treat_zero_as_filled=True,  # mets False si "0" doit compter comme vide
        show_examples=10            # nb de lignes fautives à afficher (échantillon)
    )
    
    # Voir le tableau récap si besoin :
    # excl_summary.head(20)


🚨 Exclusivité violée — base='indemnite de logement', année=2015 : 42 lignes en anomalie (sur 2278823).
MATRICULE||CODE_ORGANISME SOURCE_FICHIER indemnite de logement (code 263) indemnite de logement (code 271) indemnite de logement (code 265)
                350537E22    012015.xlsx                            40000                            40000                             None
                351478Q22    012015.xlsx                            40000                           280000                             None
                365120B42    012015.xlsx                            40000                            40000                             None
                382643D22    012015.xlsx                            40000                             5000                             None
                350537E22    022015.xlsx                            40000                            40000                             None
                351478Q22    022015.xlsx                

## 4.4 Faire une vérification pour voir si la sommme pour chaque ligne à problème donne le montant brute pour les colonnes répétés


In [11]:
bad_idx = get_multi_filled_row_indices(
    panel_solde_df,
    source_col="SOURCE_FICHIER",
    treat_zero_as_filled=True
)
print("Nombre de lignes multi-remplies (toutes bases/années confondues):", len(bad_idx))


NameError: name 'get_multi_filled_row_indices' is not defined

In [17]:

# 3) Ne sommer QUE sur ces lignes, et comparer à MONTANT BRUT
bilan_sum_all = check_sum_all_base_vs_reference_on_index(
    panel_solde_df,
    bad_idx,
    reference_col="MONTANT BRUT",
    abs_tol=1e-6,
    rel_tol=0.0,
    show_examples=10
)

🔎 Lignes testées: 286 | Anomalies (somme ≠ MONTANT BRUT): 0



## 4.5 Consolider les colonnes répétées (1ère non nulle) et supprimer les sources

Il faut sommer montant pour avoir une seule colonne si les colonnes répétés donnent le Montant brut 

In [18]:
panel_solde_df = consolidate_all_years_and_drop_sources(
    panel_solde_df,
    source_col="SOURCE_FICHIER",
    mode="sum",               # ← ICI on somme les colonnes doublons
    sep=" | ",                # ignoré en mode "sum"
    treat_zero_as_filled=True,# ignoré en mode "sum"
    new_name_fmt="{base}"     # une seule colonne finale par base
)
print(f"Après consolidation: {len(panel_solde_df):,} lignes, {panel_solde_df.shape[1]} colonnes")


Après consolidation: 15,081,954 lignes, 183 colonnes


## 4.6 Contrôle rapide : plus de colonnes '(code ...)'

In [19]:
res_codes = [c for c in panel_solde_df.columns if "(code" in c.lower()]
print("Colonnes '(code ...)' restantes :", res_codes)   # doit être []

Colonnes '(code ...)' restantes : []


## 4.7 Charger panel_solde_df_1 (panel 1)

In [23]:
panel_solde_df_1 = read_parquet_from_s3(
    BUCKET, KEY_PANEL_1,
    endpoint_url=ENDPOINT_URL,
    aws_access_key_id=AWS_KEY,
    aws_secret_access_key=AWS_SECRET,
    verify=VERIFY_SSL
)
print(f"panel_solde_df_1: {len(panel_solde_df_1):,} lignes, {panel_solde_df_1.shape[1]} colonnes")

panel_solde_df_1: 15,627,963 lignes, 40 colonnes


**Remarque**

La base panel_solde_df_1 a  15,627,963 lignes alors  que la base panel_solde_df a 15,081,954 lignes

## 4.8 Fusion sur une année donnée Et Enregistrement des non matchs dans un fichier excel

### 4.8.1 Fusion sur l'année 2015

In [16]:
merged_2015 = merge_union_for_year_with_suffix(
    panel_solde_df, panel_solde_df_1,
    year=2015,
    join_on_month=True,          # (YEAR, MONTH, MATRICULE)
    suffix_left="_b0",
    suffix_right="_b1",
    dedup_right=True,
    export_unmatched_prefix="bilan_merge",   # => crée bilan_merge_2015_left_only.xlsx / right_only.xlsx
    xlsx_chunk_rows=1_000_000,               # pour éviter la limite Excel
    xlsx_engine="xlsxwriter"
)


Fusion OUTER 2015 (join_on_month=True) → 2,305,311 lignes, 236 colonnes

Bilan _merge :
_merge
both          2278182
right_only      26488
left_only         641
Name: count, dtype: int64
XLSX écrits :
   bilan_merge_2015_left_only.xlsx
   bilan_merge_2015_right_only.xlsx


### 4.8.2 Fusion sur l'année 2016

In [17]:
merged_2016 = merge_union_for_year_with_suffix(
    panel_solde_df, panel_solde_df_1,
    year=2016,
    join_on_month=True,          # (YEAR, MONTH, MATRICULE)
    suffix_left="_b0",
    suffix_right="_b1",
    dedup_right=True,
    export_unmatched_prefix="bilan_merge",   # => crée bilan_merge_2015_left_only.xlsx / right_only.xlsx
    xlsx_chunk_rows=1_000_000,               # pour éviter la limite Excel
    xlsx_engine="xlsxwriter"
)


Fusion OUTER 2016 (join_on_month=True) → 2,428,508 lignes, 236 colonnes

Bilan _merge :
_merge
both          2195520
right_only     232297
left_only         691
Name: count, dtype: int64
XLSX écrits :
   bilan_merge_2016_left_only.xlsx
   bilan_merge_2016_right_only.xlsx


### 4.8.3 Fusion sur l'année 2017

In [18]:
merged_2017 = merge_union_for_year_with_suffix(
    panel_solde_df, panel_solde_df_1,
    year=2017,
    join_on_month=True,          # (YEAR, MONTH, MATRICULE)
    suffix_left="_b0",
    suffix_right="_b1",
    dedup_right=True,
    export_unmatched_prefix="bilan_merge",   # => crée bilan_merge_2015_left_only.xlsx / right_only.xlsx
    xlsx_chunk_rows=1_000_000,               # pour éviter la limite Excel
    xlsx_engine="xlsxwriter"
)


Fusion OUTER 2017 (join_on_month=True) → 2,510,492 lignes, 236 colonnes

Bilan _merge :
_merge
both          2474710
right_only      34886
left_only         896
Name: count, dtype: int64
XLSX écrits :
   bilan_merge_2017_left_only.xlsx
   bilan_merge_2017_right_only.xlsx


### 4.8.4 Fusion sur l'année 2018

In [19]:
merged_2018 = merge_union_for_year_with_suffix(
    panel_solde_df, panel_solde_df_1,
    year=2018,
    join_on_month=True,          # (YEAR, MONTH, MATRICULE)
    suffix_left="_b0",
    suffix_right="_b1",
    dedup_right=True,
    export_unmatched_prefix="bilan_merge",   # => crée bilan_merge_2015_left_only.xlsx / right_only.xlsx
    xlsx_chunk_rows=1_000_000,               # pour éviter la limite Excel
    xlsx_engine="xlsxwriter"
)


Fusion OUTER 2018 (join_on_month=True) → 2,669,894 lignes, 236 colonnes

Bilan _merge :
_merge
both          2603334
right_only      65494
left_only        1066
Name: count, dtype: int64
XLSX écrits :
   bilan_merge_2018_left_only.xlsx
   bilan_merge_2018_right_only.xlsx


### 4.8.5 Fusion sur l'année 2019

In [20]:
merged_2019 = merge_union_for_year_with_suffix(
    panel_solde_df, panel_solde_df_1,
    year=2019,
    join_on_month=True,          # (YEAR, MONTH, MATRICULE)
    suffix_left="_b0",
    suffix_right="_b1",
    dedup_right=True,
    export_unmatched_prefix="bilan_merge",   # => crée bilan_merge_2015_left_only.xlsx / right_only.xlsx
    xlsx_chunk_rows=1_000_000,               # pour éviter la limite Excel
    xlsx_engine="xlsxwriter"
)


Fusion OUTER 2019 (join_on_month=True) → 2,795,680 lignes, 236 colonnes

Bilan _merge :
_merge
both          2684338
right_only     111241
left_only         101
Name: count, dtype: int64
XLSX écrits :
   bilan_merge_2019_left_only.xlsx
   bilan_merge_2019_right_only.xlsx


### 4.8.6 Fusion sur l'année 2020

In [24]:
merged_2020= merge_union_for_year_with_suffix(
    panel_solde_df, panel_solde_df_1,
    year=2020,
    join_on_month=True,          # (YEAR, MONTH, MATRICULE)
    suffix_left="_b0",
    suffix_right="_b1",
    dedup_right=True,
    export_unmatched_prefix="bilan_merge",   # => crée bilan_merge_2015_left_only.xlsx / right_only.xlsx
    xlsx_chunk_rows=1_000_000,               # pour éviter la limite Excel
    xlsx_engine="xlsxwriter"
)


Fusion OUTER 2020 (join_on_month=True) → 2,922,599 lignes, 236 colonnes

Bilan _merge :
_merge
both          2841349
right_only      80124
left_only        1126
Name: count, dtype: int64
XLSX écrits :
   bilan_merge_2020_left_only.xlsx
   bilan_merge_2020_right_only.xlsx
